# Example Notebook - Prep a bladder segmentation from DICOM

Load one ultrasound DICOM, prepare the selected frame using the model's expected preprocessing, run the bladder segmentation model, and view the predicted mask over the image.

**Workflow:** choose the DICOM → preprocess its frame → load the model → predict → view the mask and overlay.


## Dependencies

Install the notebook dependencies (`numpy`, `pydicom`, `Pillow`, `TensorFlow`, and `tf_keras`) in the active Jupyter environment. Make the DICOM file and the trained `bladder.h5` model available to the notebook.

This example uses frame **15**, crops pixels `[40:605, 155:955]`, resizes to **256 × 256** with nearest-neighbor interpolation, and scales pixel values to `[0, 1]`.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Any, Dict

import numpy as np
import pydicom
from PIL import Image
from IPython.display import display

# Use legacy Keras for compatibility 
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")

FRAME_INDEX = 15 # Index of the frame to process
INPUT_SIZE = (256, 256) # Resize input images to this size
MASK_THRESHOLD = 0.4 # Threshold for mask binarization (choose values > 0.35) 
OVERLAY_COLOR = (144, 238, 144)  # Personalize Color for overlay 
OVERLAY_ALPHA = 0.45  # Transparency level for overlay
_RESAMPLE_NEAREST = Image.Resampling.NEAREST if hasattr(Image, "Resampling") else Image.NEAREST

## 1. Load the DICOM

Enter the DICOM filename when prompted. The cell reads the dataset and makes its pixel data available for preprocessing.


In [ ]:
def load_dicom(path: Path) -> pydicom.dataset.Dataset:
    return pydicom.dcmread(str(path), force=True)


dicom_path = Path(input("DICOM filename: ").strip())
dicom = load_dicom(dicom_path)
frame_count = int(getattr(dicom, "NumberOfFrames", 1) or 1)

## 2. Prepare the model input

Select frame 15, apply the crop and resize used by this example, then normalize the grayscale pixels and add a batch dimension. The final array has the shape expected for a single model input.


In [ ]:
frame = np.squeeze(dicom.pixel_array[FRAME_INDEX])
cropped_frame = frame[40:605, 155:955]
input_image = Image.fromarray(cropped_frame).resize(
    INPUT_SIZE,
    resample=_RESAMPLE_NEAREST,
)
image_arr = np.expand_dims(
    np.asarray(input_image, dtype=np.float32) / 255.0,
    axis=0,
)

display(input_image)

## 3. Load the trained model

Enter the model filename when prompted. The helper functions below register the model's custom loss/metric names and apply compatibility adjustments for the saved model format. 

Model Metrics - Dice, BCE Dice Loss, Jaccard index defined for model object 

In [ ]:
def _bladder_custom_objects() -> Dict[str, Any]:
    from tf_keras import backend as K
    from tf_keras.losses import binary_crossentropy

    def dice_coef(y_true, y_pred, smooth=1e-6):
        y_true = K.cast(y_true, "float32")
        y_pred = K.cast(y_pred, "float32")
        y_true_bin = K.greater(y_true, 0.5)
        y_pred_bin = K.greater(y_pred, 0.5)
        intersection = K.sum(K.cast(y_true_bin & y_pred_bin, "float32"))
        union = K.sum(K.cast(y_true_bin | y_pred_bin, "float32"))
        return (2.0 * intersection + smooth) / (K.sum(y_true_bin) + K.sum(y_pred_bin) + smooth)

    def bce_dice_loss(y_true, y_pred):
        return binary_crossentropy(y_true, y_pred) + (1.0 - dice_coef(y_true, y_pred))

    def jaccard_index(y_true, y_pred, smooth=1e-6):
        y_true = K.cast(y_true, "float32")
        y_pred = K.cast(y_pred, "float32")
        y_true_bin = K.greater(y_true, 0.5)
        y_pred_bin = K.greater(y_pred, 0.5)
        intersection = K.sum(K.cast(y_true_bin & y_pred_bin, "float32"))
        union = K.sum(K.cast(y_true_bin | y_pred_bin, "float32"))
        return (intersection + smooth) / (union + smooth)

    return {
        "dice_coef": dice_coef,
        "bce_dice_loss": bce_dice_loss,
        "jaccard_index": jaccard_index,
        "dice": dice_coef,
        "bcer": binary_crossentropy,
    }


def load_bladder_model(model_path: Path):
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found at {model_path}.")

    apply_unet_compat_patches()

    import tf_keras
    from tf_keras.models import load_model

    custom_objects = dict(tf_keras.utils.get_custom_objects())
    custom_objects.update(_bladder_custom_objects())

    return load_model(str(model_path), compile=False, custom_objects=custom_objects)



def apply_unet_compat_patches() -> None:
    """Minimal patches needed to deserialize the older-format bladder.h5
    UNet under a newer tf_keras/TensorFlow version (same fixes as
    app/keras_compat.py::apply_unet_compat_patches)."""
    from tf_keras.layers import Conv2DTranspose, InputLayer

    original_input = InputLayer.from_config

    @classmethod
    def input_from_config(cls, config):  # type: ignore[no-untyped-def]
        config = dict(config)
        if "batch_shape" in config and "input_shape" not in config:
            batch_shape = config.pop("batch_shape")
            if batch_shape:
                config["input_shape"] = tuple(batch_shape[1:])
        _normalize_dtype(config)
        return original_input(config)

    InputLayer.from_config = input_from_config  # type: ignore[assignment]

    original_deconv = Conv2DTranspose.from_config

    @classmethod
    def deconv_from_config(cls, config):  # type: ignore[no-untyped-def]
        config = dict(config)
        _normalize_dtype(config)
        _normalize_initializer(config, "kernel_initializer")
        _normalize_initializer(config, "bias_initializer")
        config.pop("output_padding", None)
        return original_deconv(config)

    Conv2DTranspose.from_config = deconv_from_config  # type: ignore[assignment]

def _normalize_dtype(config: dict) -> None:
    dtype = config.get("dtype")
    if isinstance(dtype, dict):
        inner = dtype.get("config", {})
        if isinstance(inner, dict) and "name" in inner:
            config["dtype"] = inner["name"]


def _normalize_initializer(config: dict, field: str) -> None:
    value = config.get(field)
    if isinstance(value, dict) and "class_name" in value:
        mapping = {
            "GlorotUniform": "glorot_uniform",
            "Zeros": "zeros",
            "HeNormal": "he_normal",
            "Ones": "ones",
        }
        config[field] = mapping.get(value["class_name"], "glorot_uniform")

In [ ]:
model_path = Path(input("Model filename (for example, bladder.h5): ").strip())
model = load_bladder_model(model_path)

## 4. Run prediction and view the result

Run inference on the preprocessed image. Pixels above the 0.4 threshold form the binary segmentation mask. The final cell displays, in order, the model input, the predicted mask, and the mask overlaid in light green on the image.


In [ ]:
prediction = np.squeeze(model.predict(image_arr))
mask = (prediction > MASK_THRESHOLD).astype(np.uint8) * 255
mask_image = Image.fromarray(mask)

base_image = input_image.convert("RGB")
color_image = Image.new("RGB", base_image.size, OVERLAY_COLOR)
tinted_image = Image.blend(base_image, color_image, OVERLAY_ALPHA)
overlay_image = Image.composite(tinted_image, base_image, mask_image)

display(input_image, mask_image, overlay_image)